# Установка и импорты

In [ ]:
! pip install openai python-dotenv pydantic

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import os
from pydantic import BaseModel
import json
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from pathlib import Path
import time
import csv
import re


from __future__ import annotations
import asyncio
import random
from dataclasses import dataclass
from typing import Any

from openai import AsyncOpenAI, APIConnectionError, APIStatusError, RateLimitError
from tqdm.asyncio import tqdm_asyncio

load_dotenv("../doc_2026-04-27_19-27-23.env", override=True)

BASE_URL = os.getenv("BASE_URL")
API_KEY = os.getenv("API_KEY")
MODEL_NAME = "Qwen3.5-9B"
TEMPERATURE = 0
MAX_TOKENS = 3000

## Загрузка словаря 

In [5]:
def load_drug_terms(csv_path: str = "illegal_terms_dictionary_edit.csv") -> str:
    seen: set[tuple[str, str | None]] = set()
    items: list[dict] = []

    with open(csv_path, encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            term = (row.get("normalized_term") or "").strip()
            cat_raw = (row.get("category") or "").strip()
            if not term:
                continue

            term = re.sub(r"\([^)]*\)", "", term)
            if "(" in term or ")" in term or len(term) > 25:
                continue
            term = term.strip(" .,:;").lower()
            if len(term) < 2:
                continue

            category: str | None = cat_raw if cat_raw else None
            key = (term, category)
            if key in seen:
                continue
            seen.add(key)
            items.append({"term": term, "category": category})

    items.sort(key=lambda x: (x["term"], x["category"] or ""))
    return json.dumps(items, ensure_ascii=False)


def load_drug_terms_short(csv_path: str = "illegal_terms_dictionary_edit.csv") -> str:
    seen: set[str] = set()
    items: list[dict] = []

    with open(csv_path, encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            cat_raw = (row.get("category") or "").strip()
            if cat_raw != "drugs":
                continue

            term = (row.get("normalized_term") or "").strip()
            if not term:
                continue

            term = re.sub(r"\([^)]*\)", "", term)
            if "(" in term or ")" in term or len(term) > 25:
                continue
            term = term.strip(" .,:;").lower()
            if len(term) < 2:
                continue

            if term in seen:
                continue
            seen.add(term)
            items.append({"term": term, "category": "drugs"})

    items.sort(key=lambda x: x["term"])
    return json.dumps(items, ensure_ascii=False)


DRUG_TERMS = load_drug_terms("../illegal_terms_dictionary_edit.csv")
DRUG_TERMS_SHORT = load_drug_terms_short("../illegal_terms_dictionary_edit.csv")

print(f"Загружено терминов в DRUG_TERMS: {len(json.loads(DRUG_TERMS))}")
print(f"Терминов в DRUG_TERMS_SHORT: {len(json.loads(DRUG_TERMS_SHORT))}")

Загружено терминов в DRUG_TERMS: 754
Терминов в DRUG_TERMS_SHORT: 104


## Структурированный вывод модели

In [6]:
# ============================================================
# Pydantic-схема для structured output vLLM
# ============================================================
# id принудительно подставляется из item на стороне Python (см. classify_one),
# здесь схема нужна только чтобы зафиксировать формат ответа модели.
# Минимум полей — меньше места для отклонений формата.

class DrugMentionClassification(BaseModel):
    has_drug_mention: bool

print(json.dumps(DrugMentionClassification.model_json_schema(), indent=2, ensure_ascii=False))

{
  "properties": {
    "has_drug_mention": {
      "title": "Has Drug Mention",
      "type": "boolean"
    }
  },
  "required": [
    "has_drug_mention"
  ],
  "title": "DrugMentionClassification",
  "type": "object"
}


## Промпты 

In [7]:
prompt_b = """РОЛЬ:
Ты — высокоточная система бинарной классификации (NLP-модель), предназначенная для модерации сообщений Telegram. Твоя цель — максимально точно (с приоритетом на высокий F1-score) определять наличие упоминаний наркотических и психоактивных веществ.

ЗАДАЧА:
Определи, содержит ли текст сообщения упоминания наркотиков или связанной с ними деятельности.

ФОРМАТ ВХОДА:
JSON с полями:
- id: идентификатор сообщения
- text: текст сообщения (единственный источник анализа)

ФОРМАТ ВЫХОДА:
Строго JSON: {{"has_drug_mention": true | false}}

ОПРЕДЕЛЕНИЕ КЛАССА true:
Ставь true, если выполнено ХОТЯ БЫ ОДНО из условий:

1. Прямое упоминание наркотиков:
   - каннабис, марихуана, гашиш, кокаин, героин, амфетамин, метамфетамин, экстази, LSD и т.д.
   - любые термины из словаря ниже (полное или частичное совпадение по корню)

2. Сленг, жаргон, эвфемизмы:
   - шишки, травка, соль (в наркотическом контексте), меф, спиды, колёса, бошки и т.д.
   - английский сленг: weed, coke, meth, molly, acid и т.д.

3. Намеренно искажённые слова:
   - замены символов: м@рuху@на, к0к@ин, мефедр0н
   - пробелы/разделители: "м е ф", "к о к с"
   - транслит: marikhuana, geroin, mefedron

4. Контекст действий:
   - покупка, продажа, обмен, доставка, закладки
   - употребление, хранение, производство
   - поиск: "где взять", "купить", "есть ли", "ищу"

5. Косвенные сигналы:
   - эмодзи: 💊 🌿 🍁 ❄️ 🔥 🚬 💉
   - сочетание нейтральных слов с подозрительным контекстом

6. Частично неоднозначные случаи:
   - если есть разумное подозрение на наркотический контекст → true

СЛОВАРЬ ТЕРМИНОВ НАРКОТИЧЕСКОЙ ТЕМАТИКИ (список объектов с term и category — используй как опорный список):
{DRUG_TERMS}

ОПРЕДЕЛЕНИЕ КЛАССА false:
Ставь false, если:

1. Упоминания отсутствуют полностью
2. Слова-омонимы используются в бытовом значении:
   - "соль", "сахар", "таблетки" без контекста наркотиков
3. Лекарства:
   - если это медицинский контекст без признаков злоупотребления
4. Явная ирония или метафоры:
   - "я подсел на кофе как на наркотик"
5. Общие разговоры без связи с наркотиками

ПРИОРИТЕТ ПРИНЯТИЯ РЕШЕНИЯ (ВАЖНО ДЛЯ F1):
1. Контекст > отдельные слова
2. Словарь (DRUG_TERMS) > эвристики
3. Лучше false, чем ложный true ТОЛЬКО если нет достаточных сигналов
4. Но при слабом, но логичном подозрении → true

АЛГОРИТМ (внутренний):
1. Нормализуй текст:
   - приведи к нижнему регистру
   - убери спецсимволы
   - учти замены символов и транслит
2. Проверь совпадения со словарём DRUG_TERMS (term и его category помогут понять контекст: drugs/drug_trade/drug_effect/instruments/general/crypto_payment)
3. Проверь сленг и искажения
4. Проанализируй контекст (действия + объекты)
5. Учти эмодзи
6. Прими бинарное решение

ОГРАНИЧЕНИЯ:
- Используй ТОЛЬКО поле text
- НЕ добавляй объяснений
- НЕ добавляй новых полей
- НЕ изменяй структуру JSON
- Ответ ДОЛЖЕН быть валидным JSON

ПРИМЕР:

ВХОД:
{{
  "id": "abc__0",
  "text": "где купить меф?"
}}

ВЫХОД:
{{"has_drug_mention": true}}

ВХОДНОЙ JSON ДЛЯ КЛАССИФИКАЦИИ:
{INPUT_JSON}
"""

In [8]:
prompt_с = """РОЛЬ:
Ты — опытный помощник по классификации сообщений в чат-ботах Telegram для задачи модерации контента, связанного с покупкой, продажей, перепродажей, обменом наркотических средств. Модерация контента необходима для поддержания имиджа компании, выпускающей чат-ботов.

ЗАДАЧА:
Определи, есть ли в сообщении упоминание наркотических средств и психогенных веществ.

КОНТЕКСТ:
На вход подаётся JSON с текстом сообщения бота. Нужно выполнить бинарную классификацию:
- true — если в тексте есть упоминание наркотиков, наркотических веществ, сленговых названий наркотиков, их покупки, продажи, употребления, хранения, изготовления или распространения
- false — если таких упоминаний нет

Дополнительные правила:
- Учитывай, что написание наркотиков может быть намеренно видоизменено посредством замены части символов в слове
- Анализируй смайлики, которые могут указывать на то, что в сообщении обсуждаются наркотики
- Обращай внимание как на русские, так и на английские названия наркотиков
- Учитывай, что названия наркотиков часто могут быть сленговыми

ОПОРНЫЙ СЛОВАРЬ СЛЕНГОВЫХ И НАРКОТИЧЕСКИХ ТЕРМИНОВ (список объектов с term и category="drugs"):
{DRUG_TERMS_SHORT}

ОГРАНИЧЕНИЯ:
- Анализируй только текст из входного поля
- Не добавляй объяснений вне JSON
- Верни только JSON, строго по заданной структуре
- Если упоминание неоднозначное, но разумно связано с наркотиками, ставь true
- Учитывай, что названия наркотиков могут быть омонимичны словам из бытовой речи, не связанной с наркотиками (например, соль)
- Не считай упоминания лекарств наркотиками, если в тексте нет явной связи с наркотическими веществами
- Не присваивай true явно ироничному употреблению названий наркотиков
- Не игнорируй сленговые названия наркотиков

ПРИМЕРЫ:

ПРИМЕР ВХОДА:
{{
  "id": "abc__0",
  "text": "где купить наркотики?"
}}

ПРИМЕР ВЫХОДА:
{{"has_drug_mention": true}}

Теперь обработай входной JSON и верни только выходной JSON по той же структуре.

ВХОДНОЙ JSON:
{INPUT_JSON}
"""

In [9]:
prompt_d = """РОЛЬ:
Ты — высокоточная система бинарной классификации (NLP-модель), предназначенная для модерации сообщений Telegram. Твоя цель — максимально точно (с приоритетом на высокий F1-score) определять наличие упоминаний наркотических и психоактивных веществ.

ЗАДАЧА:
Определи, содержит ли текст сообщения упоминания наркотиков или связанной с ними деятельности.

ФОРМАТ ВХОДА:
JSON с полями:
- id: идентификатор сообщения
- text: текст сообщения (единственный источник анализа)

ФОРМАТ ВЫХОДА:
Строго JSON: {{"has_drug_mention": true | false}}

ОПРЕДЕЛЕНИЕ КЛАССА true:
Ставь true, если выполнено ХОТЯ БЫ ОДНО из условий:

1. Прямое упоминание наркотиков:
   - каннабис, марихуана, гашиш, кокаин, героин, амфетамин, метамфетамин, экстази, LSD и т.д.
   - любые термины из словаря ниже (полное или частичное совпадение по корню)

2. Прямое упоминание сущностей, связанных с наркотиками и наркоторговлей:
   - обменник, клад, кладмен, фасовка, закладка и т.д.

2. Сленг, жаргон, эвфемизмы:
   - шишки, травка, соль (в наркотическом контексте), меф, спиды, колёса, бошки и т.д.
   - английский сленг: weed, coke, meth, molly, acid и т.д.

3. Намеренно искажённые слова, ососбенно в названиях каналов и ботов через @:
   - замены символов: м@рuху@на, к0к@ин, мефедр0н
   - добавление лишних символов: DeaIler
   - пробелы/разделители: "м е ф", "к о к с"
   - транслит: marikhuana, geroin, mefedron

4. Контекст действий:
   - покупка, продажа, обмен, доставка, закладки
   - употребление, хранение, производство
   - поиск: "где взять", "купить", "есть ли", "ищу"

5. Подозрительные аббревиатуры и одиночные буквы латиницей в качестве вопроса:
   - Bbgg, Sh, I, CV GK j

6. Упоминания криптокошельков и криптовалют

7. Фразы с двойным дном и иносказания:
   -  Главное не забывать: счастье — это когда ты нашёл, а тебя нет!

8. Косвенные сигналы:
   - эмодзи: 💊 🌿 🍁 ❄️ 🔥 🚬 💉
   - сочетание нейтральных слов с подозрительным контекстом

9. Частично неоднозначные случаи:
   - если есть разумное подозрение на наркотический контекст → true

СЛОВАРЬ ТЕРМИНОВ НАРКОТИЧЕСКОЙ ТЕМАТИКИ (список объектов с term и category — используй как опорный список):
{DRUG_TERMS}

ОПРЕДЕЛЕНИЕ КЛАССА false:
Ставь false, если:

1. Упоминания отсутствуют полностью
2. Слова-омонимы используются в бытовом значении:
   - "соль", "сахар", "таблетки" без контекста наркотиков
3. Лекарства:
   - если это медицинский контекст без признаков злоупотребления
4. Явная ирония или метафоры:
   - "я подсел на кофе как на наркотик"
5. Общие разговоры без связи с наркотиками

ПРИОРИТЕТ ПРИНЯТИЯ РЕШЕНИЯ (ВАЖНО ДЛЯ F1):
1. Контекст > отдельные слова
2. Словарь (DRUG_TERMS) > эвристики
3. Лучше false, чем ложный true ТОЛЬКО если нет достаточных сигналов
4. Но при слабом, но логичном подозрении → true

АЛГОРИТМ (внутренний):
1. Нормализуй текст:
   - приведи к нижнему регистру
   - убери спецсимволы
   - учти замены символов и транслит
2. Проверь совпадения со словарём DRUG_TERMS (term и его category помогут понять контекст: drugs/drug_trade/drug_effect/instruments/general/crypto_payment)
3. Проверь сленг и искажения
4. Проанализируй контекст (действия + объекты)
5. Учти эмодзи
6. Прими бинарное решение

ОГРАНИЧЕНИЯ:
- Используй ТОЛЬКО поле text
- НЕ добавляй объяснений
- НЕ добавляй новых полей
- НЕ изменяй структуру JSON
- Ответ ДОЛЖЕН быть валидным JSON

ПРИМЕР:

ВХОД:
{{
  "id": "abc__0",
  "text": "где купить меф?"
}}

ВЫХОД:
{{"has_drug_mention": true}}

ВХОДНОЙ JSON ДЛЯ КЛАССИФИКАЦИИ:
{INPUT_JSON}
"""

## Сборщик промптов

In [10]:
SYSTEM_PROMPT = "Ты - помощник по классификации текста для задачи модерации на предмет упоминания наркотиков."

def build_messages_b(item):
    user_content = prompt_b.format(
        DRUG_TERMS=DRUG_TERMS,
        INPUT_JSON=json.dumps(item, ensure_ascii=False, indent=2),
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
    ]

def build_messages_c(item):
    user_content = prompt_с.format(
        DRUG_TERMS_SHORT=DRUG_TERMS_SHORT,
        INPUT_JSON=json.dumps(item, ensure_ascii=False, indent=2),
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
    ]

def build_messages_d(item):
    user_content = prompt_d.format(
        DRUG_TERMS=DRUG_TERMS,
        INPUT_JSON=json.dumps(item, ensure_ascii=False, indent=2),
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
    ]
PROMPT_BUILDERS = {
    "b": build_messages_b,
    "c": build_messages_c,
    "d": build_messages_d,
}


## Классификация с логированием 

In [ ]:
# ============================================================
# Конфигурация
# ============================================================

MAX_CONCURRENCY = 32
MAX_RETRIES = 5
REQUEST_TIMEOUT = 250.0
MAX_TEXT_LEN = 30000
RESPONSE_MAX_TOKENS = 25000        # с запасом под thinking mode Qwen
TEMPERATURE = 0.0

INPUT_PARQUET = "../test.parquet"


# ============================================================
# Подготовка данных: составной id session_id__turn_idx
# ============================================================

def build_input_jsons(df: pd.DataFrame) -> list[dict[str, str]]:
    df = df.reset_index(drop=True).copy()
    df["turn_idx"] = df.groupby("session_id").cumcount()

    items: list[dict[str, str]] = []
    for _, row in df.iterrows():
        text = f"Вопрос: {row['question']}\nОтвет: {row['answer']}"
        items.append({
            "id": f"{row['session_id']}__{row['turn_idx']}",
            "text": text[:MAX_TEXT_LEN],
        })

    ids = [it["id"] for it in items]
    assert len(set(ids)) == len(ids), "Составной id не уникален — проверьте данные."
    return items


# ============================================================
# Структура результата
# ============================================================

@dataclass
class Prediction:
    id: str
    has_drug_mention: bool | None
    error: str | None

    def to_dict(self) -> dict[str, Any]:
        return {
            "id": self.id,
            "has_drug_mention": self.has_drug_mention,
            "error": self.error,
        }


# ============================================================
# Логгер запросов
# ============================================================

import hashlib
from datetime import datetime, timezone


class RequestLogger:
    """Асинхронно-безопасный построчный логгер в JSONL."""

    def __init__(self, path: str) -> None:
        self.path = path
        self._lock = asyncio.Lock()

    async def log(self, record: dict[str, Any]) -> None:
        async with self._lock:
            with open(self.path, "a", encoding="utf-8") as f:
                f.write(json.dumps(record, ensure_ascii=False) + "\n")


def _extract_user_prompt(messages: list[dict[str, str]]) -> str:
    """Берём контент user-сообщения. Если их несколько — соединяем переводами строк."""
    parts = [m["content"] for m in messages if m.get("role") == "user"]
    return "\n\n".join(parts)


# ============================================================
# Один запрос к LLM с ретраями, structured output и логированием
# ============================================================

async def classify_one(
    client: AsyncOpenAI,
    item: dict[str, str],
    messages: list[dict[str, str]],
    semaphore: asyncio.Semaphore,
    logger: RequestLogger,
    prompt_version: str,
) -> Prediction:
    user_prompt = _extract_user_prompt(messages)
    prompt_sha = hashlib.sha256(user_prompt.encode("utf-8")).hexdigest()

    async with semaphore:
        last_err: str | None = None
        for attempt in range(MAX_RETRIES):
            t_start = time.perf_counter()
            ts_iso = datetime.now(timezone.utc).isoformat()

            log_record: dict[str, Any] = {
                "timestamp": ts_iso,
                "id": item["id"],
                "prompt_version": prompt_version,
                "attempt": attempt,
                "elapsed_time": None,
                "prompt_sha256": prompt_sha,
                "response": None,
                "parsed": None,
                "error": None,
                "status": None,
            }

            try:
                resp = await client.chat.completions.parse(
                    model=MODEL_NAME,
                    messages=messages,
                    temperature=TEMPERATURE,
                    max_tokens=RESPONSE_MAX_TOKENS,
                    response_format=DrugMentionClassification,
                    extra_body={"chat_template_kwargs": {"enable_thinking": True}},
                    timeout=REQUEST_TIMEOUT,
                )
                elapsed = time.perf_counter() - t_start
                raw_content = resp.choices[0].message.content
                parsed = resp.choices[0].message.parsed

                log_record["elapsed_time"] = round(elapsed, 3)
                log_record["response"] = raw_content

                if parsed is None:
                    log_record["status"] = "parse_failed"
                    log_record["error"] = "parsed is None (модель не вернула валидный JSON)"
                    await logger.log(log_record)
                    last_err = log_record["error"]
                else:
                    log_record["status"] = "ok"
                    log_record["parsed"] = bool(parsed.has_drug_mention)
                    await logger.log(log_record)
                    return Prediction(
                        id=item["id"],
                        has_drug_mention=bool(parsed.has_drug_mention),
                        error=None,
                    )

            except (APIConnectionError, RateLimitError, asyncio.TimeoutError) as e:
                elapsed = time.perf_counter() - t_start
                log_record["elapsed_time"] = round(elapsed, 3)
                log_record["status"] = "transient_error"
                log_record["error"] = f"{type(e).__name__}: {e}"
                await logger.log(log_record)
                last_err = log_record["error"]

            except APIStatusError as e:
                elapsed = time.perf_counter() - t_start
                log_record["elapsed_time"] = round(elapsed, 3)
                err_text = f"APIStatusError {e.status_code}: {e}"
                if 500 <= e.status_code < 600:
                    log_record["status"] = "transient_error"
                    log_record["error"] = err_text
                    await logger.log(log_record)
                    last_err = err_text
                else:
                    # 4xx — финал, не ретраим
                    log_record["status"] = "fatal_error"
                    log_record["error"] = err_text
                    await logger.log(log_record)
                    return Prediction(id=item["id"], has_drug_mention=None, error=err_text)

            except Exception as e:
                elapsed = time.perf_counter() - t_start
                log_record["elapsed_time"] = round(elapsed, 3)
                log_record["status"] = "transient_error"
                log_record["error"] = f"{type(e).__name__}: {e}"
                await logger.log(log_record)
                last_err = log_record["error"]

            backoff = min(2 ** attempt + random.uniform(0, 1), 30.0)
            await asyncio.sleep(backoff)

        return Prediction(id=item["id"], has_drug_mention=None,
                          error=f"max retries exceeded: {last_err}")


# ============================================================
# Возобновляемость: какие id уже успешно обработаны
# ============================================================

def load_done_ids(jsonl_path: str) -> set[str]:
    done: set[str] = set()
    if not os.path.exists(jsonl_path):
        return done
    with open(jsonl_path, encoding="utf-8") as f:
        for line in f:
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue
            if isinstance(rec.get("has_drug_mention"), bool):
                done.add(rec["id"])
    return done


# ============================================================
# Запуск одного промпта над всем датасетом
# ============================================================

async def run_prompt(
    items: list[dict[str, str]],
    prompt_version: str,
    output_path: str,
    log_path: str,
) -> None:
    builder = PROMPT_BUILDERS[prompt_version]

    done_ids = load_done_ids(output_path)
    todo = [it for it in items if it["id"] not in done_ids]
    print(f"[{prompt_version}] всего: {len(items)} | уже готово: {len(done_ids)} | к обработке: {len(todo)}")
    print(f"[{prompt_version}] лог запросов: {log_path}")

    if not todo:
        return

    client = AsyncOpenAI(base_url=BASE_URL, api_key=API_KEY)
    semaphore = asyncio.Semaphore(MAX_CONCURRENCY)
    file_lock = asyncio.Lock()
    logger = RequestLogger(log_path)

    async def worker(item: dict[str, str]) -> Prediction:
        messages = builder(item)
        pred = await classify_one(
            client=client,
            item=item,
            messages=messages,
            semaphore=semaphore,
            logger=logger,
            prompt_version=prompt_version,
        )
        async with file_lock:
            with open(output_path, "a", encoding="utf-8") as f:
                f.write(json.dumps(pred.to_dict(), ensure_ascii=False) + "\n")
        return pred

    tasks = [worker(it) for it in todo]
    await tqdm_asyncio.gather(*tasks, desc=f"classify [{prompt_version}]")
    await client.close()


# ============================================================
# main
# ============================================================

async def main() -> None:
    df = pd.read_parquet(INPUT_PARQUET).reset_index(drop=True)
    items = build_input_jsons(df)
    print(f"Подготовлено {len(items)} объектов, уникальных id: {len({i['id'] for i in items})}\n")

    for version in ["d"]:
        output_path = f"predictions_{version}.jsonl"
        log_path = f"requests_{version}.log.jsonl"
        await run_prompt(items, version, output_path, log_path)

        n_ok, n_err = 0, 0
        with open(output_path, encoding="utf-8") as f:
            for line in f:
                rec = json.loads(line)
                if isinstance(rec.get("has_drug_mention"), bool):
                    n_ok += 1
                else:
                    n_err += 1
        print(f"[{version}] итог: успешно {n_ok}, ошибок {n_err}\n")


await main()

Подготовлено 748 объектов, уникальных id: 748

[d] всего: 748 | уже готово: 748 | к обработке: 0
[d] лог запросов: requests_d.log.jsonl
[d] итог: успешно 748, ошибок 0



## Метрики 

In [ ]:
def load_predictions(jsonl_path: str) -> pd.DataFrame:
    """Читает predictions_*.jsonl и разворачивает составной id."""
    rows = []
    with open(jsonl_path, encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    pred_df = pd.DataFrame(rows)
    parts = pred_df["id"].str.rsplit("__", n=1, expand=True)
    pred_df["session_id"] = parts[0]
    pred_df["turn_idx"] = parts[1].astype(int)
    return pred_df


def evaluate_results(jsonl_path: str, df_truth: pd.DataFrame, label: str):
    """
    Мерджим truth и предсказания по (session_id, turn_idx) — составной ключ,
    потому что в одной сессии несколько турнов.
    """
    if not Path(jsonl_path).exists():
        print(f"[{label}] файл {jsonl_path} не найден, пропускаю")
        return None

    df_pred = load_predictions(jsonl_path)
    if df_pred.empty:
        print(f"[{label}] файл пустой, пропускаю")
        return None

    # Дубли по id (например, после повторного запуска) — оставляем последний.
    df_pred = df_pred.drop_duplicates(subset="id", keep="last")

    # Считаем ошибочные предсказания отдельно, чтобы они не портили метрики.
    n_failed = df_pred["has_drug_mention"].isna().sum()
    if n_failed:
        print(f"[{label}] {n_failed} объектов с ошибкой LLM — исключаю из метрик")
    df_pred = df_pred[df_pred["has_drug_mention"].notna()].copy()

    # Готовим truth с тем же составным ключом.
    df_truth_local = df_truth.reset_index(drop=True).copy()
    df_truth_local["session_id"] = df_truth_local["session_id"].astype(str)
    df_truth_local["turn_idx"] = df_truth_local.groupby("session_id").cumcount()
    df_pred["session_id"] = df_pred["session_id"].astype(str)

    df_merged = df_truth_local.merge(
        df_pred[["session_id", "turn_idx", "has_drug_mention"]],
        on=["session_id", "turn_idx"],
        how="inner",
    )

    if len(df_merged) == 0:
        print(f"[{label}] нет совпадений по (session_id, turn_idx), пропускаю.")
        return None
    if len(df_merged) != len(df_truth_local):
        print(f"[{label}] предупреждение: смержилось {len(df_merged)} из {len(df_truth_local)} строк truth")

    y_true = (df_merged["message_label"] == "illegal").astype(int)
    y_pred = df_merged["has_drug_mention"].astype(int)

    metrics = {
        "version": label,
        "n": len(df_merged),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }

    print(f"\n=== Промпт {label} (n={metrics['n']}) ===")
    print(f"  accuracy : {metrics['accuracy']:.4f}")
    print(f"  precision: {metrics['precision']:.4f}")
    print(f"  recall   : {metrics['recall']:.4f}")
    print(f"  f1       : {metrics['f1']:.4f}")

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    print("  confusion matrix [строки=truth, столбцы=pred]:")
    print(pd.DataFrame(cm, index=["truth_legal", "truth_illegal"],
                       columns=["pred_legal", "pred_illegal"]))

    print("\n  classification_report:")
    print(classification_report(y_true, y_pred, target_names=["legal", "illegal"], zero_division=0))
    return metrics


df_truth = pd.read_parquet("../test.parquet").reset_index(drop=True)

all_metrics = []
for version in ["d"]:
    m = evaluate_results(f"predictions_{version}.jsonl", df_truth, version)
    if m is not None:
        all_metrics.append(m)

if all_metrics:
    print("\n=== Сводная таблица ===")
    print(pd.DataFrame(all_metrics).set_index("version").round(4))


=== Промпт d (n=748) ===
  accuracy : 0.9225
  precision: 0.8069
  recall   : 0.9631
  f1       : 0.8782
  confusion matrix [строки=truth, столбцы=pred]:
               pred_legal  pred_illegal
truth_legal           481            50
truth_illegal           8           209

  classification_report:
              precision    recall  f1-score   support

       legal       0.98      0.91      0.94       531
     illegal       0.81      0.96      0.88       217

    accuracy                           0.92       748
   macro avg       0.90      0.93      0.91       748
weighted avg       0.93      0.92      0.92       748


=== Сводная таблица ===
           n  accuracy  precision  recall      f1
version                                          
d        748    0.9225     0.8069  0.9631  0.8782


## Логи ошибок 

In [12]:
import json
from collections import Counter

for v in ["b", "c"]:
    path = f"../predictions_{v}.jsonl"
    errors = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            if rec.get("has_drug_mention") is None:
                errors.append(rec.get("error", "<no error field>"))

    print(f"\n=== {v}: {len(errors)} ошибок ===")
    # Группируем по первым 200 символам, чтобы увидеть типы ошибок
    counter = Counter(e[:200] if e else "<empty>" for e in errors)
    for msg, cnt in counter.most_common(5):
        print(f"  [{cnt}×] {msg}")


=== b: 0 ошибок ===

=== c: 1 ошибок ===
  [1×] max retries exceeded: LengthFinishReasonError: Could not parse response content as the length limit was reached - CompletionUsage(completion_tokens=10000, prompt_tokens=2394, total_tokens=12394, compl


## Анализ ошибок

In [ ]:
PREVIEW_LEN = 800        # сколько символов question/answer показывать
SHOW_N = 100              # сколько примеров каждого типа ошибки выводить
SAVE_CSV = True          # сохранять ли все ошибки в CSV для ручного разбора


def build_error_frame(jsonl_path: str, df_truth: pd.DataFrame, label: str) -> pd.DataFrame | None:
    if not Path(jsonl_path).exists():
        print(f"[{label}] файл {jsonl_path} не найден, пропускаю")
        return None

    df_pred = load_predictions(jsonl_path)          # уже определён в ячейке с метриками
    if df_pred.empty:
        print(f"[{label}] файл пустой, пропускаю")
        return None

    df_pred = df_pred.drop_duplicates(subset="id", keep="last")
    df_pred = df_pred[df_pred["has_drug_mention"].notna()].copy()

    df_t = df_truth.reset_index(drop=True).copy()
    df_t["session_id"] = df_t["session_id"].astype(str)
    df_t["turn_idx"] = df_t.groupby("session_id").cumcount()
    df_pred["session_id"] = df_pred["session_id"].astype(str)

    df = df_t.merge(
        df_pred[["session_id", "turn_idx", "has_drug_mention"]],
        on=["session_id", "turn_idx"],
        how="inner",
    )
    if len(df) == 0:
        print(f"[{label}] нет совпадений по (session_id, turn_idx), пропускаю.")
        return None

    df["y_true"] = (df["message_label"] == "illegal").astype(int)
    df["y_pred"] = df["has_drug_mention"].astype(int)

    def _err_type(r):
        if r["y_true"] == 1 and r["y_pred"] == 0:
            return "FN"   # пропустили наркотики -> бьёт по recall
        if r["y_true"] == 0 and r["y_pred"] == 1:
            return "FP"   # ложная тревога -> бьёт по precision
        return "OK"

    df["error_type"] = df.apply(_err_type, axis=1)
    df["prompt_version"] = label
    return df


def show_errors(df: pd.DataFrame, error_type: str, n: int = SHOW_N) -> None:
    sub = df[df["error_type"] == error_type]
    total = (df["error_type"] != "OK").sum()
    print(f"\n{'='*72}")
    print(f"  {error_type}: {len(sub)} шт. (всего ошибок: {total}, объектов: {len(df)})")
    print(f"{'='*72}")
    for _, r in sub.head(n).iterrows():
        print(f"\n--- {r['session_id']}__{r['turn_idx']}   (true={r['y_true']}, pred={r['y_pred']}) ---")
        print(f"  Вопрос: {str(r['question'])[:PREVIEW_LEN]}")
        print(f"  Ответ : {str(r['answer'])[:PREVIEW_LEN]}")
    if len(sub) > n:
        print(f"\n  ... ещё {len(sub) - n} примеров (см. CSV или увеличьте SHOW_N)")


# df_truth уже загружен в ячейке с метриками; перечитаем на случай отдельного запуска
df_truth = pd.read_parquet("../test.parquet").reset_index(drop=True)

for version in ["b", "c"]:
    err_df = build_error_frame(f"../predictions_{version}.jsonl", df_truth, version)
    if err_df is None:
        continue

    n_fn = (err_df["error_type"] == "FN").sum()
    n_fp = (err_df["error_type"] == "FP").sum()
    print(f"\n\n########## ПРОМПТ {version}: FN={n_fn} (бьют по recall), FP={n_fp} (бьют по precision) ##########")

    # FN важнее для полноты — показываем первыми
    show_errors(err_df, "FN")
    show_errors(err_df, "FP")

    if SAVE_CSV:
        out = err_df[err_df["error_type"] != "OK"][
            ["session_id", "turn_idx", "error_type", "y_true", "y_pred", "question", "answer"]
        ]
        out_path = f"errors_{version}.csv"
        out.to_csv(out_path, index=False, encoding="utf-8")
        print(f"\n[{version}] все {len(out)} ошибок сохранены в {out_path}")



########## ПРОМПТ b: FN=81 (бьют по recall), FP=2 (бьют по precision) ##########

  FN: 81 шт. (всего ошибок: 83, объектов: 748)

--- telegram-8412110593-adyoika-8412110593-NOn-52042546487-0945453745.08a6ae68-4dce-bf35-bd1c-5c9ab37bc449__0   (true=1, pred=0) ---
  Вопрос: Баланс 0  gel
  Ответ : оператор @GeorgDeaIler Бот  https://t.me/georgevip_bot новости  https://t.me/AjikaShop24

--- telegram-8412110593-adyoika-8412110593-NOn-52042546487-5554805867.b339136f-e357-c4aa-5621-ddb5bb28552c__0   (true=1, pred=0) ---
  Вопрос: Hvg
  Ответ : оператор @GeorgDeaIler Бот  https://t.me/georgevip_bot новости  https://t.me/AjikaShop24

--- telegram-8412110593-adyoika-8412110593-NOn-52042546487-9542764540.c674085d-a4f5-1de1-5754-a63e0162af54__0   (true=1, pred=0) ---
  Вопрос: I
  Ответ : оператор @GeorgDeaIler Бот  https://t.me/georgevip_bot новости  https://t.me/AjikaShop24

--- a67d25b2-f026-0a83-fa52-c9ef7514f2df.4d23324c-ec6e-886e-b963-f779c49c5c14__0   (true=1, pred=0) ---
  Вопрос: Перев

##  Число токенов 

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-9B")

def count_tokens(text: str) -> int:
    return len(tokenizer.encode(text))

count_tokens(prompt_b)